<a href="https://colab.research.google.com/github/assassindiv/Small-LM/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import sentencepiece as spm

#--- training tokenozer ---#

spm.SentencePieceTrainer.train(
    input="wiki.train.tokens",
    model_prefix="bpe",
    vocab_size=5000,
    model_type="bpe",
    bos_id=1,
    eos_id=2,
    pad_id=0,
    unk_id=3,
)

sp=spm.SentencePieceProcessor()
sp.load("bpe.model")

#---loading---#
with open("wiki.train.tokens", "r")as f:
  text=f.read()
  data=torch.tensor(
      sp.encode(text,add_bos=True,add_eos=True),
      dtype=torch.long
  )
  vocab_size=sp.get_piece_size()
  print("tokens",data.size(0))
  print("vocab:",vocab_size)

In [ ]:
def load_checkpoint(model, optimizer, path):
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    step = checkpoint["step"]
    print(f"Checkpoint loaded from step {step}")
    return step


In [ ]:
import os

checkpoint_dir = "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

def save_checkpoint(model, optimizer, step, val_loss=None):
    path = os.path.join(checkpoint_dir, f"checkpoint_{step}.pt")
    torch.save({
        "step": step,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_loss": val_loss
    }, path)
    print(f"Checkpoint saved at step {step} | val_loss: {val_loss:.4f}")



In [ ]:
block_size=256
batch_size=12

n=int(0.9 * len(data))
train_data=data[:n]
val_data=data[n:]

def get_batch(split):
  source=train_data if split=='train'else val_data
  ix=torch.randint(0,len(source)-block_size,(batch_size,))
  x=torch.stack([source[i:i+block_size]for i in ix])
  y=torch.stack([source[i+1:i+block_size+1]for i in ix])
  return x.to(device), y.to(device)


In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        norm = x.pow(2).mean(dim=-1, keepdim=True)
        return x * torch.rsqrt(norm + self.eps) * self.weight


In [ ]:
def apply_rope(q, k, start_pos):
    # q, k: [B, H, T, D]
    B, H, T, D = q.shape
    device = q.device
    assert D % 2 == 0

    theta = 10000 ** (-torch.arange(0, D, 2, device=device).float() / D)
    pos = torch.arange(start_pos, start_pos + T, device=device).float()

    freqs = torch.einsum("t,d->td", pos, theta)
    cos = freqs.cos()[None, None, :, :]
    sin = freqs.sin()[None, None, :, :]

    q1, q2 = q[..., ::2], q[..., 1::2]
    k1, k2 = k[..., ::2], k[..., 1::2]

    q = torch.cat([q1 * cos - q2 * sin,
                   q1 * sin + q2 * cos], dim=-1)

    k = torch.cat([k1 * cos - k2 * sin,
                   k1 * sin + k2 * cos], dim=-1)

    return q, k


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import math

class SelfAttention(nn.Module):
    def __init__(self, embed_dim, n_heads):
        super().__init__()
        assert embed_dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads

        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x,past_kv=None):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        past_len = 0 if past_kv is None else past_kv[0].size(2)
        q, k = apply_rope(q, k, past_len )

        if past_kv is not None:
          past_k,past_v=past_kv
          k=torch.cat([past_k,k],dim=2)
          v=torch.cat([past_v,v],dim=2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        total_len=k.size(2)
        mask = torch.tril(torch.ones(T, total_len, device=x.device)).unsqueeze(0).unsqueeze(0)

        att = att.masked_fill(mask == 0, float("-inf"))

        att = F.softmax(att, dim=-1)

        out = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out), (k.detach(), v.detach())




In [ ]:
class FeedForward(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.GELU(),
            nn.Linear(4 * embed_dim, embed_dim)
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
class Block(nn.Module):
    def __init__(self, embed_dim, n_heads):
        super().__init__()
        self.ln1 = nn.RMSNorm(embed_dim)
        self.ln2 = nn.RMSNorm(embed_dim)
        self.attn = SelfAttention(embed_dim, n_heads)
        self.ff = FeedForward(embed_dim)

    def forward(self, x):
        attn_out, _ = self.attn(self.ln1(x), past_kv=None)
        x = x + attn_out
        x = x + self.ff(self.ln2(x))
        return x

    def forward_infer(self, x, past_kv):
        attn_out, kv = self.attn(self.ln1(x), past_kv)
        x = x + attn_out
        x = x + self.ff(self.ln2(x))
        return x, kv


In [ ]:
class TransformerLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed_dim = 192
        self.n_layers = 8
        self.n_heads = 8

        self.token_emb = nn.Embedding(vocab_size, self.embed_dim)


        self.blocks = nn.ModuleList(
            [Block(self.embed_dim, self.n_heads) for _ in range(self.n_layers)]
        )

        self.ln_f = nn.RMSNorm(self.embed_dim)
        self.lm_head = nn.Linear(self.embed_dim, vocab_size, bias=False)


        self.lm_head.weight = self.token_emb.weight

    def forward(self, x, targets=None):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)

        h = self.token_emb(x)
        for block in self.blocks:
            h = block(h)

        h = self.ln_f(h)
        logits = self.lm_head(h)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(B*T, vocab_size),
                targets.view(B*T)
            )
        return logits, loss

    @torch.no_grad()
    def forward_infer(self, x, past_kv):
        B, T = x.shape
        past_len = 0 if past_kv[0] is None else past_kv[0][0].size(2)

        pos = torch.arange(past_len, past_len + T, device=x.device)
        pos = pos % block_size

        h = self.token_emb(x)

        new_kv = []
        for block, layer_past in zip(self.blocks, past_kv):
            h, kv = block.forward_infer(h, layer_past)
            new_kv.append(kv)

        h = self.ln_f(h)
        logits = self.lm_head(h)
        return logits, new_kv


In [ ]:
@torch.no_grad()
def evaluate_val_loss(model, val_data, block_size, batch_size=12):
    model.eval()
    total_loss = 0.0
    n_batches = 0
    for _ in range(10):
        ix = torch.randint(0, len(val_data) - block_size, (batch_size,))
        x = torch.stack([val_data[i:i+block_size] for i in ix]).to(device)
        y = torch.stack([val_data[i+1:i+block_size+1] for i in ix]).to(device)

        _, loss = model(x, y)
        total_loss += loss.item()
        n_batches += 1

    model.train()
    return total_loss / n_batches


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = TransformerLM().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

for step in range(100001):
    x, y = get_batch("train")
    logits, loss = model(x, y)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if step % 5000 == 0:
        val_loss = evaluate_val_loss(model, val_data, block_size)
        print(f"step {step} | train_loss: {loss.item():.3f} | val_loss: {val_loss:.3f}")
        save_checkpoint(model, optimizer, step, val_loss)


In [ ]:
def sample_top_k(logits, k=50):
    values, indices = torch.topk(logits, k)
    probs = F.softmax(values, dim=-1)
    next_token = indices.gather(-1, torch.multinomial(probs, 1))
    return next_token



In [ ]:
@torch.no_grad()
def generate(prompt, max_new_tokens=100, temperature=1.0, top_k=50):
    model.eval()

    x = torch.tensor(
        [sp.encode(prompt, add_bos=True)],
        dtype=torch.long,
        device=device
    )

    past_kv = [None] * model.n_layers

    for step in range(max_new_tokens):
        x_cond = x[:, -1:]

        logits, past_kv = model.forward_infer(x_cond, past_kv)


        t = max(0.7, temperature * (1 - step / max_new_tokens))
        logits = logits[:, -1, :] / t
        for token_id in x[0].tolist()[-50:]:
              logits[:, token_id] *= 0.8


        next_id = sample_top_k(logits, k=top_k)

        x = torch.cat([x, next_id], dim=1)

    return sp.decode(x[0].tolist())


In [ ]:
print(generate("A human in the", 200, temperature=1.0, top_k=40))